# Data Understanding & Target Leakage Analysis

## Objective

The purpose of this notebook is to understand the structure of the loan dataset and identify **potential target leakage** before performing any exploratory data analysis or modeling.

In credit risk modeling, it is critical that the model only uses information that is available **at the time of loan application**. Any variables that capture post-loan outcomes (such as repayments or recoveries) can unintentionally leak information about the target variable and lead to overly optimistic and unrealistic model performance.

This notebook focuses on:
- Understanding the target variable (loan default)
- Identifying post-loan and outcome-based features
- Removing leakage-prone variables to create a model-safe dataset


In [1]:
# Core libraries
import pandas as pd
import numpy as np


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)


In [3]:
df = pd.read_csv("loan_data.csv")
df.shape


(887379, 30)

#### Target Variable Identification

The objective of this project is to predict whether a loan will default or not.

In this dataset, the column loan_condition represents the final repayment status of the loan and will be used as the target variable.

This makes the problem a binary classification task, where the model learns to distinguish between defaulted and non-defaulted loans.

In [4]:
df["loan_condition"].value_counts()


loan_condition
Good Loan    819950
Bad Loan      67429
Name: count, dtype: int64

#### Target Class Definition

For the purpose of this project, the target variable loan_condition is interpreted as follows:

Good Loan (0) → Loan was fully repaid and did not default

Bad Loan (1) → Borrower failed to repay the loan (default)

This mapping aligns with real-world credit risk modeling, where predicting defaults (Bad Loans) is the primary business objective.

In [5]:
df["loan_default"] = df["loan_condition"].map({
    "Good Loan": 0,
    "Bad Loan": 1
})

df["loan_default"].value_counts()


loan_default
0    819950
1     67429
Name: count, dtype: int64

#### What is Data Leakage?

Data leakage occurs when a model is trained using information that would not be available at the time a prediction is made.

In credit risk modeling, this typically happens when features contain information that is only known after the loan has been approved or after repayment has occurred.

Using such features leads to unrealistically high model performance that does not generalize to real-world scenarios.

#### When is the prediction made?

In this project, the model is intended to predict loan default at the time of loan approval.

This means:

Only information available on or before the loan issue date (issue_d) can be used as model features.

Any variables that are calculated or updated after loan issuance or during repayment must be excluded from model training.

Establishing this timeline is essential to correctly identify and prevent data leakage.

In [6]:
# Columns that are likely to cause data leakage (post-loan information)
leakage_columns = [
    "total_pymnt",
    "total_rec_prncp",
    "recoveries",
    "installment",
    "final_d"
]

leakage_columns


['total_pymnt', 'total_rec_prncp', 'recoveries', 'installment', 'final_d']

## Data Leakage & Feature Selection Decision

In credit risk modeling, it is critical that the model only uses information that would be
available **at the time of loan application or approval**.

Based on business understanding and domain knowledge, the following columns contain
post-loan or outcome-related information and therefore must be excluded from model training:

### Leakage-Prone Columns
- `total_pymnt` – Total amount paid by the borrower (known only after repayments)
- `total_rec_prncp` – Principal repaid over time
- `recoveries` – Amount recovered after default (directly reveals default outcome)
- `installment` – Derived from loan terms and interest; may encode repayment behavior
- `final_d` – Loan closure date (future information)

### Decision
These columns will be **removed before modeling** to prevent data leakage and to ensure
the model reflects real-world decision-making conditions.

This step ensures that model performance is realistic, interpretable, and suitable for
production use.


In [7]:
# List of leakage-prone columns identified earlier
leakage_cols = [
    "total_pymnt",
    "total_rec_prncp",
    "recoveries",
    "installment",
    "final_d"
]

# Drop leakage columns
df_clean = df.drop(columns=leakage_cols)

# Verify shape after dropping
df_clean.shape


(887379, 26)

## Data Understanding Summary

In this phase, we developed a strong understanding of the business problem and the dataset.

Key outcomes:
- Defined the prediction objective as **loan default prediction at the time of loan approval**
- Identified the target variable: `loan_condition`
- Examined class distribution and confirmed class imbalance
- Established a clear prediction timeline to prevent data leakage
- Identified and removed post-loan variables that leak outcome information
- Created a clean dataset (`df_clean`) suitable for exploratory data analysis and modeling

This structured approach ensures that all subsequent analysis and models
reflect real-world credit decision scenarios and produce reliable, interpretable results.
